# Coming from Brieflow

[Brieflow](https://github.com/cheeseman-lab/brieflow) runs an optical pooled screen from raw images to clustered genes as a Snakemake workflow.
The page below runs Brieflow's aggregate and cluster stages with mantispy, in the order Brieflow runs them, on a slice of Aconcagua.
Aconcagua is Brieflow's own reanalysis of a HeLa CRISPR screen, and its gene-level outputs are public, so each step below compares the mantispy result with the one Brieflow published.

The Brieflow code is cited at release 1.5.1, commit [`61457ae`](https://github.com/cheeseman-lab/brieflow/tree/61457ae1e50392bdf70852f4a87544fea4474e88), and every function name links to that commit.
The published outputs were made with Brieflow 1.4.10, as the screen's [metadata file](https://ops-explorer-public.s3.amazonaws.com/aconcagua/aconcagua/metadata/experimental_metadata.yaml) records.

## The slice and the published outputs

{func}`~mantispy.ds.aconcagua_cells` holds the single cells of eight genes and a small sample of the non-targeting control cells, with Brieflow's measurements under mantispy's feature names.
Brieflow runs every stage once per cell class ([`split_datasets`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/rules/aggregate.smk#L33)), so the page takes the interphase cells. Run it again with `"Mitotic"` for the mitotic cells.
The published file is Brieflow's interphase gene table, with one row per gene and one per non-targeting guide, its features, the bootstrap p-values, and the PHATE and Leiden results.

In [1]:
import warnings

import anndata as ad
import igraph as ig
import leidenalg
import numpy as np
import pandas as pd
import pooch
import scanpy as sc
from scipy import stats
from sklearn.metrics import adjusted_rand_score

import mantispy as mt

warnings.filterwarnings("ignore", category=FutureWarning)

published = ad.read_h5ad(
    pooch.retrieve(
        "https://ops-explorer-public.s3.amazonaws.com/aconcagua/aconcagua/visualizations/interphase/aggregated_data.h5ad",
        known_hash="sha256:f55756939edc8ddf108092d88c4efe248765b4db4ec39322d4b4a988e04af929",
        fname="brief_interphase.h5ad",
    )
)
cells = mt.ds.aconcagua_cells()
adata = cells[cells.obs["Metadata_Class"] == "Interphase"].copy()
GENES = sorted(set(adata.obs["Metadata_Gene"]) - {"nontargeting"})
in_slice = int((adata.obs["Metadata_Control_Type"] == "negcon").sum())
in_brieflow = int(published.obs.loc[published.obs_names.str.startswith("nontargeting"), "n_cells"].sum())
print(f"published features: {published.n_vars}")
print(f"control cells: {in_slice} in the slice, {in_brieflow} aggregated by Brieflow ({in_slice / in_brieflow:.1%})")
pd.DataFrame(
    {
        "cells in the slice": adata.obs["Metadata_Gene"].value_counts()[GENES],
        "cells Brieflow aggregated": published.obs.loc[GENES, "n_cells"],
    }
).T

published features: 76
control cells: 22241 in the slice, 681919 aggregated by Brieflow (3.3%)


,DTL,HAUS6,ITGAV,PSMB7,RRM1,SF3B3,SKA2,TLN1
cells in the slice,1781,1524,1823,1845,1821,1790,766,1881
cells Brieflow aggregated,2661,1717,4411,3079,4198,4896,773,4214


The slice holds a few percent of Brieflow's control cells. For SKA2 and HAUS6 it holds almost every cell Brieflow aggregated, and for the other genes between about a third and two thirds of them, drawn at random.
The comparisons below are limited by that sampling, not only by the methods.

## Label the cells

Brieflow's [`perturbation_filter`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/filter.py#L159) drops cells with no gene, and [`prepare_alignment_data`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/align.py#L17) renames each non-targeting cell after its guide, as `nontargeting_<barcode>`.
Every non-targeting guide is then a row of its own in the gene table and in the clustering, where the controls would otherwise collapse into one row.
`Metadata_Label` below holds that name, and `Metadata_Batch` the plate and well, which is Brieflow's batch.
Drop the cells with no gene before building the label, or they become controls.

In [2]:
adata = adata[adata.obs["Metadata_Gene"].notna()].copy()
is_control = (adata.obs["Metadata_Control_Type"] == "negcon").to_numpy()
adata.obs["Metadata_Label"] = np.where(
    is_control, "nontargeting_" + adata.obs["Metadata_sgRNA"].astype(str), adata.obs["Metadata_Gene"].astype(str)
)
adata.obs["Metadata_Batch"] = adata.obs["Metadata_Plate"].astype(str) + "_" + adata.obs["Metadata_Well"].astype(str)
print(f"{adata.n_obs} cells, {adata.obs['Metadata_Label'].nunique()} labels")

35472 cells, 258 labels


## Filter

Brieflow's [`filter`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/rules/aggregate.smk#L61) rule runs once per plate, well and cell class.
[`missing_values_filter`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/filter.py#L188) drops the features missing in at least 10% of a well's cells (`drop_cols_threshold`), then the cells missing at least 1% of the features (`drop_rows_threshold`), and fills what is left with a KNN imputer of five neighbours.
A feature dropped in any well is lost from the whole screen when the wells are pooled.
[`intensity_filter`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/filter.py#L300) then fits a local outlier factor on the mean intensity of each channel in each object and drops the most outlying 1% of each well's cells.

In [3]:
batch = adata.obs["Metadata_Batch"].to_numpy()
print(f"missing values: {int(np.isnan(adata.X).sum())}")

missing = pd.DataFrame(np.isnan(adata.X), columns=adata.var_names).groupby(batch).mean()
adata = adata[:, ~(missing >= 0.1).any().to_numpy()].copy()
adata = adata[np.isnan(adata.X).mean(axis=1) < 0.01].copy()
mt.pp.impute(adata, method="knn", n_neighbors=5, by="Metadata_Batch")

channels = ["DAPI", "TUBULIN", "GH2AX", "PHALLOIDIN"]  # Brieflow's phenotype.channel_names
adata.var["intensity_mean"] = adata.var_names.str.endswith(tuple(f"_MeanIntensity_{c}" for c in channels))
mt.pp.outliers(adata, method="lof", contamination=0.01, key="intensity_mean", by="Metadata_Batch")
flagged = adata.obs.groupby("Metadata_Gene", observed=True)["qc_outlier"].mean().rename("share of cells flagged")
adata = adata[~adata.obs["qc_outlier"]].copy()
flagged.to_frame().T.round(3)

missing values: 0


Metadata_Gene,DTL,HAUS6,ITGAV,PSMB7,RRM1,SF3B3,SKA2,TLN1,nontargeting
share of cells flagged,0.018,0.007,0.013,0.011,0.046,0.022,0.004,0.018,0.006


The public cell table holds the cells Brieflow kept after this filter, so the slice has no missing values and the first three lines change nothing here.
The outlier filter still drops cells, because it always drops the most outlying 1% of each well.
It drops more of some genes than of the controls, since a knockout that changes the intensity of a stain, such as RRM1's gamma-H2AX, moves its cells to the edge of the intensity distribution.
Read the share above before trusting a strong intensity phenotype, or pass a smaller `contamination`.

Where cells are missing values, {func}`~mantispy.pp.impute` differs from Brieflow in how it measures the distance between cells. It uses every feature, where Brieflow uses only the features with missing values and draws its donors from cells that are complete in those.

## Feature table

Brieflow's [`generate_feature_table`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/scripts/aggregate/generate_feature_table.py) centres and scales every feature on the non-targeting cells of each plate and well ([`centerscale_on_controls`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/align.py#L266)), takes the median of each guide's cells, and then the median of each gene's guide medians.
Its `feature_normalization` setting picks the scaling, either `"standard"` for the mean and standard deviation, the [default](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/rules/aggregate.smk#L113), or `"mad"` for the median and the median absolute deviation, which is {func}`~mantispy.pp.normalize` with `method="mad_robustize"`.
The cell below builds the table both ways.

In [4]:
by = ["Metadata_Plate", "Metadata_Well"]
mt.pp.normalize(adata, method="standardize", by=by, reference="negcon", key_added="controls_standard")
mt.pp.normalize(adata, method="mad_robustize", by=by, reference="negcon", key_added="controls_mad")


def feature_table(layer):
    """Brieflow's gene table: the median of each guide's cells, then the median of each label's guides."""
    guides = mt.tl.aggregate(adata, by=["Metadata_Label", "Metadata_sgRNA"], func="median", min_cells=1, layer=layer)
    table = mt.tl.aggregate(guides, by=["Metadata_Label"], func="median", min_cells=1)
    table.obs_names = table.obs["Metadata_Label"].astype(str).to_numpy()
    return table


tables = {"standard": feature_table("controls_standard"), "mad": feature_table("controls_mad")}
print(
    f"{tables['mad'].n_obs} rows, all in the published table: {tables['mad'].obs_names.isin(published.obs_names).all()}"
)

WARNING  normalize: 34 of 1483 features have no reference spread in some ['Metadata_Plate', 'Metadata_Well'] group 
         and are NaN there

WARNING  normalize: 62 of 1483 features have no reference spread in some ['Metadata_Plate', 'Metadata_Well'] group 
         and are NaN there

258 rows, all in the published table: True


The published file keeps the features Brieflow's default feature list ([`get_feature_table_cols`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/cell_data_utils.py#L338)) selects for the bootstrap, which are the intensity, shape and correlation features of the nucleus and the cell.
The function below renames them to mantispy's names, and the comparison correlates each gene's 76 values with Brieflow's.

In [5]:
OBJECTS = {"nucleus": "Nuclei", "cell": "Cells"}
INTENSITY = {
    "integrated": "IntegratedIntensity",
    "mean": "MeanIntensity",
    "mean_edge": "MeanIntensityEdge",
    "std_edge": "StdIntensityEdge",
    "mass_displacement": "MassDisplacement",
}
SHAPE = {"area": "Area", "form_factor": "FormFactor", "solidity": "Solidity", "eccentricity": "Eccentricity"}


def mantispy_name(name):
    """The mantispy name of one of Brieflow's 76 published features."""
    obj, rest = name.split("_", 1)
    obj = OBJECTS[obj]
    if rest in SHAPE:
        return f"{obj}_AreaShape_{SHAPE[rest]}"
    if rest.startswith("correlation_"):
        _, a, b = rest.split("_")
        return f"{obj}_Correlation_Correlation_{a}_{b}"
    channel, stat = rest.split("_", 1)
    if stat.startswith("mean_frac_"):
        return f"{obj}_RadialDistribution_MeanFrac_{channel}_{int(stat[-1]) + 1}of4"
    return f"{obj}_Intensity_{INTENSITY[stat]}_{channel}"


FEATURES = pd.Series(published.var_names.map(mantispy_name), index=published.var_names)
brieflow = pd.DataFrame(np.asarray(published.X), index=published.obs_names, columns=published.var_names)


def as_published(table):
    """A table's 76 published features, under Brieflow's names."""
    return pd.DataFrame(np.asarray(table[:, FEATURES.to_numpy()].X), index=table.obs_names, columns=published.var_names)


pd.DataFrame(
    {
        scaling: {gene: np.corrcoef(as_published(table).loc[gene], brieflow.loc[gene])[0, 1] for gene in GENES}
        for scaling, table in tables.items()
    }
).T.round(3)

,DTL,HAUS6,ITGAV,PSMB7,RRM1,SF3B3,SKA2,TLN1
standard,0.922,0.114,0.929,0.904,0.968,0.822,0.181,0.923
mad,0.999,0.975,0.998,0.997,0.999,0.991,0.988,0.997


With the median and MAD scaling every gene's profile matches Brieflow's, which says Aconcagua was run with `feature_normalization: mad`.
With Brieflow's default scaling the strong genes still correlate, but the genes with a weak interphase phenotype do not.
Check the setting in your own configuration before comparing numbers.

The non-targeting rows agree less well, because each control guide in the slice keeps a few percent of its cells.

In [6]:
controls = [label for label in tables["mad"].obs_names if label.startswith("nontargeting")]
ours = as_published(tables["mad"]).loc[controls]
correlation = np.median([np.corrcoef(ours.loc[c], brieflow.loc[c])[0, 1] for c in controls])
print(f"median correlation over {len(controls)} control guides: {correlation:.3f}")
for layer in ["controls_standard", "controls_mad"]:
    flagged = adata.var.loc[FEATURES.to_numpy(), f"degenerate_scale_{layer}"]
    print(f"{layer}: {int(flagged.sum())} of the {len(FEATURES)} features have no spread among some well's controls")

median correlation over 250 control guides: 0.266
controls_standard: 0 of the 76 features have no spread among some well's controls
controls_mad: 0 of the 76 features have no spread among some well's controls


Brieflow centres a feature with no spread among a well's controls but doesn't scale it, while {func}`~mantispy.pp.normalize` sets it to missing in that well and flags it in `var`.
None of the published features is flagged, as the cell above prints, and the PCA step below drops the flagged features.

## Feature bootstrap

Brieflow's [`run_gene_bootstrap`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/bootstrap.py#L428) tests each feature of each gene against medians resampled from the control guides' cells, on the feature table above, and [`apply_multiple_hypothesis_correction`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/bootstrap.py#L651) adds Benjamini-Hochberg q-values over every gene and feature.
Without a `bootstrap_features_fp` file it tests the default feature list ([`prepare_bootstrap_data`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/scripts/aggregate/prepare_bootstrap_data.py#L98)), the features above, so subset the object to them first.
{func}`~mantispy.tl.bootstrap_features` computes the null exactly, where Brieflow simulates it 100,000 times ([`num_sims`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/rules/aggregate.smk#L390)).

In [7]:
tested = adata[:, FEATURES.to_numpy()].copy()
mt.tl.bootstrap_features(
    tested,
    groupby="Metadata_Gene",
    replicate="Metadata_sgRNA",
    reference="negcon",
    control_scope="pooled",
    layer="controls_mad",
)
boot = tested.uns["mantispy"]["bootstrap"]
brieflow_p = pd.DataFrame(published.layers["p_values"], index=published.obs_names, columns=published.var_names)
brieflow_name = dict(zip(FEATURES, FEATURES.index, strict=True))
boot["brieflow_pvalue"] = [
    brieflow_p.at[gene, brieflow_name[feature]] for gene, feature in zip(boot["group"], boot["feature"], strict=True)
]
print(f"Spearman correlation of the p-values: {stats.spearmanr(boot['pvalue'], boot['brieflow_pvalue'])[0]:.3f}")
print(
    f"p-values of zero: {int((boot['pvalue'] == 0).sum())} from mantispy, {int((boot['brieflow_pvalue'] == 0).sum())} from Brieflow"
)
pd.crosstab(
    boot["pvalue"] < 0.05,
    boot["brieflow_pvalue"] < 0.05,
    rownames=["mantispy p < 0.05"],
    colnames=["Brieflow p < 0.05"],
)

Spearman correlation of the p-values: 0.849
p-values of zero: 110 from mantispy, 370 from Brieflow


Brieflow p < 0.05,False,True
mantispy p < 0.05,,
False,136,107
True,0,365


Every feature mantispy calls at p < 0.05 Brieflow calls too.
The features only Brieflow calls likely come from the slice's smaller samples, since a gene with half its cells has a wider null and each control guide keeps a few percent of its cells, and from how the two count ties, below.
Compare p-values rather than q-values here, because Brieflow's q-values correct over every gene of the screen and these over eight.

mantispy also differs on purpose in how it treats p-values at the edge of the null.
Brieflow's simulated p-value is zero whenever none of its 100,000 null medians passes the observed one, and it raises a zero to `1 / num_sims` before the correction.
mantispy computes each p-value exactly, so it is zero only when the observed median lies beyond every median the null can produce, and it keeps that zero.
Brieflow also counts a null median equal to the observed one as below it, where mantispy counts it half, so a feature that most cells share one value of is not called.

## Align

Brieflow's [`align`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/scripts/aggregate/align.py) script scores perturbations, centres and scales every cell within its plate and well, reduces with PCA, and applies TVN and CORAL.
It starts again from the filtered cells, not from the feature table.

### Perturbation score

Brieflow skips this step by default ([`skip_perturbation_score: True`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/rules/aggregate.smk#L146)), and the published file has no AUCs to compare with.
When it runs, [`perturbation_score`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/perturbation_score.py#L27) fits a logistic regression per gene, on the 200 features with the largest ANOVA F statistic, against as many control cells as the gene has, and records each cell's out-of-fold probability and each gene's ROC AUC.
It needs 100 cells for the gene and its control sample together, so `min_cells=50`.
It uses ten folds once the two reach 200 cells ([`n_splits`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/perturbation_score.py#L195)), which every gene in the slice does.
Brieflow scales on the mean and standard deviation of the controls here, whatever `feature_normalization` says.

`replicate=None` draws the folds over cells, as Brieflow does.
mantispy's default, `replicate="Metadata_sgRNA"`, keeps each guide's cells in one fold, so the model is always tested on a guide it was not trained on.
The cell runs both.

In [8]:
mt.pp.normalize(adata, method="standardize", by=by, reference=None)
degenerate = adata.var["degenerate_scale"] | adata.var["degenerate_scale_controls_standard"]
print(f"dropping {int(degenerate.sum())} features with no spread in some well")
adata = adata[:, ~degenerate.to_numpy()].copy()

for replicate, key in [(None, "perturbation_score"), ("Metadata_sgRNA", "perturbation_score_by_guide")]:
    mt.tl.perturbation_score(
        adata,
        groupby="Metadata_Gene",
        reference="negcon",
        layer="controls_standard",
        replicate=replicate,
        n_features=200,
        n_folds=10,
        min_cells=50,
        key_added=key,
    )
pd.DataFrame(
    {
        "AUC, folds over cells": adata.uns["mantispy"]["perturbation_score"].set_index("group")["auc"],
        "AUC, folds over guides": adata.uns["mantispy"]["perturbation_score_by_guide"].set_index("group")["auc"],
        "folds over guides": adata.uns["mantispy"]["perturbation_score_by_guide"].set_index("group")["n_folds"],
    }
).T.round(3)

WARNING  normalize: 34 of 1483 features have no reference spread in some ['Metadata_Plate', 'Metadata_Well'] group 
         and are NaN there

dropping 34 features with no spread in some well


<path>:7: UserWarning: replicate=None draws folds over cells, so the model can learn what one guide's cells share and score it as the group's phenotype. Pass replicate= the guide column, e.g. replicate='Metadata_sgRNA'.
  mt.tl.perturbation_score(


group,DTL,HAUS6,ITGAV,PSMB7,RRM1,SF3B3,SKA2,TLN1
"AUC, folds over cells",0.807,0.602,0.876,0.847,0.851,0.706,0.615,0.852
"AUC, folds over guides",0.798,0.586,0.866,0.824,0.837,0.635,0.563,0.841
folds over guides,4.000,4.000,4.000,4.000,4.000,4.000,4.000,4.000


The guide folds give lower AUCs, because a gene's AUC no longer gains from the model learning what one guide's cells look like.
They also cap the folds at the gene's number of guides, as the last row shows.
A gene with too few cells gets no AUC, and Brieflow's AUC filter then drops it, since a missing AUC is not above any threshold.

### PCA

The first line of the cell above is Brieflow's [`centerscale_by_batch`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/align.py#L223), which standardizes every cell within its plate and well, and the cell dropped the features that have no spread in some well.
Brieflow fits `PCA(n_components=0.99)` ([`align.py`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/scripts/aggregate/align.py#L94)) on a random sample of at most 100,000 cells, which here is every cell, and keeps the fewest components that explain 99% of the variance.
Fit the full PCA once to find that number, then refit with it, so that `varm` and `uns` describe the components kept.

In [9]:
sc.pp.pca(adata, n_comps=adata.n_vars - 1, svd_solver="covariance_eigh")
n_comps = int(np.searchsorted(np.cumsum(adata.uns["pca"]["variance_ratio"]), 0.99, side="right")) + 1
sc.pp.pca(adata, n_comps=n_comps, svd_solver="covariance_eigh")
control_obs = adata.obs[adata.obs["Metadata_Control_Type"] == "negcon"]
controls_per_well = control_obs.groupby("Metadata_Batch", observed=True).size()
print(f"{n_comps} components; {controls_per_well.min()} to {controls_per_well.max()} control cells per well")

723 components; 390 to 534 control cells per well


### TVN and CORAL

TVN (typical variation normalization) whitens the profiles so that the variation among control cells has the same size in every direction, and CORAL (correlation alignment) gives each well's controls the covariance of all the controls.
Brieflow's [`tvn_on_controls`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/align.py#L143) centres and scales the components on the controls, rotates them onto the principal components of the controls, centres and scales them again on each well's controls, and then aligns each well with CORAL.
`epsilon=0.5` is the shrinkage it adds to each covariance before inverting it.

In [10]:
mt.pp.tvn(adata, batch_key="Metadata_Batch", reference="negcon", use_rep="X_pca", epsilon=0.5)

WARNING  tvn: 46 batch(es) have no more reference rows than the 723 components ('1_A01' (534), '1_A02' (483),      
         '1_A03' (464), ...)

The slice keeps a few percent of the control cells, so every well has fewer controls than there are components, and {func}`~mantispy.pp.tvn` warns.
Brieflow skips CORAL for a well with fewer controls than components ([`align.py`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/align.py#L200)), so on this slice it would align no well, while mantispy aligns every one.
On the full screen each well has many times the controls.

## Aggregate

Brieflow's [`aggregate`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/aggregate/aggregate.py#L14) takes the median of the TVN components over all the cells of each gene, and of each non-targeting guide.
It can first drop the cells whose perturbation score is below `ps_probability_threshold`, keeping the cells with no score, and it doesn't by default.
The cells it keeps are the ones that look least like the controls, so the gene profiles move further from the controls, and closer to other strongly scored genes, than the gene's cells as a whole do.
[`clean_aggregate`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/scripts/cluster/clean_aggregate.py) then keeps the rows with more than `min_cell_cutoffs[cell_class]` cells, a setting with no default.

In [11]:
ps_probability_threshold = None  # Brieflow's default
if ps_probability_threshold is not None:
    adata = adata[~(adata.obs["perturbation_score"] < ps_probability_threshold)].copy()
genes = mt.tl.aggregate(adata, by=["Metadata_Label"], func="median", min_cells=1, use_rep="X_tvn")
genes.obs_names = genes.obs["Metadata_Label"].astype(str).to_numpy()

min_cell_cutoff = 0  # set per cell class in Brieflow's configuration
genes = genes[genes.obs["Metadata_CellCount"] > min_cell_cutoff].copy()
genes

AnnData object with n_obs × n_vars = 258 × 723
    obs: 'Metadata_Label', 'Metadata_CellCount', 'Metadata_SiteCount', 'Metadata_Gene', 'Metadata_Perturbation', 'Metadata_Perturbation_Type', 'Metadata_Control_Type', 'Metadata_Class', 'Metadata_Tier', 'Metadata_Control', 'Metadata_ObjectType', 'qc_outlier', 'qc_fail'
    var: 'object', 'feature_group', 'feature', 'channel', 'scale', 'angle', 'gray_levels', 'radial_bin', 'params', 'is_feature', 'feature_kind'
    uns: 'mantispy'
    layers: None (.X)

`min_cells=1` turns off mantispy's own minimum of 10 cells, so the cut-off is applied once.

## Cluster

Brieflow's [`phate_leiden_clustering`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/scripts/cluster/phate_leiden_clustering.py#L28) script first applies [`filter_by_perturbation_auc`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/cluster/phate_leiden_clustering.py#L308), which is off by default ([`perturbation_auc_threshold`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/rules/cluster.smk#L42)), and then embeds the rows with PHATE ([`run_phate`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/cluster/phate_leiden_clustering.py#L83), ten neighbours, cosine distance by [default](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/rules/cluster.smk#L35)) and clusters them with Leiden at each configured resolution.
The workflow doesn't filter genes on the bootstrap before clustering.
{func}`~mantispy.tl.phate` implements PHATE with the parameter names of `phate.PHATE`, and its automatic choice of the number of diffusion steps `t` can land on a different value than phate's, so pass `t=` to match a given run.

In [12]:
perturbation_auc_threshold = None  # Brieflow's default
if perturbation_auc_threshold is not None:
    auc = adata.uns["mantispy"]["perturbation_score"].set_index("group")["auc"]
    control = genes.obs["Metadata_Control_Type"] == "negcon"
    genes = genes[control | (genes.obs["Metadata_Gene"].map(auc) > perturbation_auc_threshold)].copy()

mt.tl.phate(genes, knn=10, metric="cosine", use_rep=None, seed=42)
mt.tl.cluster(genes, method="leiden", neighbors_key="phate", resolution=1.0)

Brieflow publishes the clusters of all its genes, and Leiden's clusters depend on every row in the graph, so the clusters of eight genes and the controls can't match them.
What carries over is which genes sit closest, so the cell below finds each gene's most similar other gene in the TVN profiles, beside the genes Brieflow put in the same cluster.

In [13]:
profiles = genes[GENES].X
unit = profiles / np.linalg.norm(profiles, axis=1, keepdims=True)
similarity = unit @ unit.T
np.fill_diagonal(similarity, -np.inf)
brieflow_cluster = published.obs.loc[GENES, "cluster_group_1"]
pd.DataFrame(
    {
        "nearest gene, mantispy": np.array(GENES)[similarity.argmax(axis=1)],
        "same Brieflow cluster": [
            ", ".join(g for g in GENES if g != gene and brieflow_cluster[g] == brieflow_cluster[gene]) or "none"
            for gene in GENES
        ],
    },
    index=GENES,
)

,"nearest gene, mantispy",same Brieflow cluster
DTL,RRM1,RRM1
HAUS6,SKA2,SKA2
ITGAV,TLN1,TLN1
PSMB7,DTL,none
RRM1,DTL,DTL
SF3B3,RRM1,none
SKA2,HAUS6,HAUS6
TLN1,ITGAV,ITGAV


Every gene that shares a Brieflow cluster with another of the eight has that gene as its nearest neighbour here.

mantispy also clusters on a different graph on purpose.
Brieflow runs Leiden on PHATE's diffusion operator, the affinity with each row normalized to sum to one, averaged with its transpose, and iterates Leiden until the partition stops changing ([`run_leiden_clustering`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/lib/cluster/phate_leiden_clustering.py#L127)).
{func}`~mantispy.tl.phate` stores the symmetric affinity before that normalization, in the layout {func}`scanpy.pp.neighbors` uses, and {func}`~mantispy.tl.cluster` runs {func}`scanpy.tl.leiden` on it.
The cell rebuilds Brieflow's graph from the same affinity and compares the two partitions.

In [14]:
affinity = genes.obsp["phate_connectivities"].toarray() + np.eye(genes.n_obs)  # PHATE's kernel keeps the self-loops
walk = affinity / affinity.sum(axis=1, keepdims=True)
graph = ig.Graph.Weighted_Adjacency(((walk + walk.T) / 2).tolist(), mode="undirected")
partition = leidenalg.find_partition(
    graph,
    leidenalg.RBConfigurationVertexPartition,
    weights=graph.es["weight"],
    n_iterations=-1,
    seed=42,
    resolution_parameter=1.0,
)
brieflow_graph = np.array(partition.membership).astype(str)
print(f"clusters: {genes.obs['cluster'].nunique()} on mantispy's graph, {len(set(brieflow_graph))} on Brieflow's")
print(f"adjusted Rand index between the two: {adjusted_rand_score(genes.obs['cluster'], brieflow_graph):.3f}")

clusters: 11 on mantispy's graph, 14 on Brieflow's


adjusted Rand index between the two: 0.168


The two partitions agree only in part on this slice, by the adjusted Rand index above, where most of the rows are control guides.
On Brieflow's full published gene table, the {doc}`PHATE and Leiden case study <../../case_studies/phate_leiden>` compares the route with Brieflow's own clusters against CORUM complexes and Reactome pathways, and the {doc}`optical pooled screens case study <../../case_studies/optical_pooled_screens>` works through the whole screen.
Brieflow's [`benchmark_clusters`](https://github.com/cheeseman-lab/brieflow/blob/61457ae1e50392bdf70852f4a87544fea4474e88/workflow/scripts/cluster/benchmark_clusters.py) scores the clusters against CORUM, KEGG and STRING, while {func}`~mantispy.metrics.known_relationships` scores the profiles against CORUM without a clustering.

## Reading the outputs

Read a gene's results in order.
First, check that its guides agree.
Second, check that its whole profile differs from the controls.
Third, read which features changed.

A gene's guides agree when each one moves the gene's significant features the same way, and the bootstrap's guide table holds each guide's effect.

In [15]:
guides = tested.uns["mantispy"]["bootstrap_guides"]
called = boot.loc[boot["pvalue"] < 0.05, ["group", "feature"]]
signs = (
    guides.merge(called, on=["group", "feature"])
    .groupby(["group", "feature"])["effect"]
    .agg(lambda e: np.sign(e).nunique() == 1)
)
signs.groupby("group").mean().rename("guides agree in sign").to_frame().T.round(2)

group,DTL,HAUS6,ITGAV,PSMB7,RRM1,SF3B3,SKA2,TLN1
guides agree in sign,1.0,0.75,1.0,0.98,0.98,0.8,0.61,1.0


The table gives the share of each gene's called features in which all its guides move the same way.
SKA2, HAUS6 and SF3B3, the genes with the lowest AUCs above, have the least consistent guides.

The bootstrap tests one feature at a time, so it doesn't say whether the gene differs as a whole.
{func}`~mantispy.tl.hit_calling` tests the whole profile, and with `null="pseudogenes"` it ranks each gene among pseudo-genes built from as many control guides.
Half of the control guides fit its covariance, which needs more guides than dimensions, so the cell reduces the guide profiles to 50 principal components first.

In [16]:
guide_profiles = mt.tl.aggregate(
    adata, by=["Metadata_Gene", "Metadata_sgRNA"], func="median", min_cells=1, use_rep="X_tvn"
)
sc.pp.pca(guide_profiles, n_comps=50)
mt.tl.hit_calling(guide_profiles, groupby="Metadata_Gene", use_rep="X_pca", null="pseudogenes", seed=0)
hits = guide_profiles.uns["mantispy"]["hits"]
pseudogenes = hits["group"].str.startswith("pseudogene")
print(f"pseudo-genes called: {int(hits.loc[pseudogenes, 'is_hit'].sum())} of {int(pseudogenes.sum())}")
hits.loc[~pseudogenes, ["group", "pvalue", "qvalue", "is_hit"]]

pseudo-genes called: 0 of 31


,group,pvalue,qvalue,is_hit
0,DTL,0.000880,0.001173,True
1,HAUS6,0.119613,0.119613,False
2,ITGAV,0.000880,0.001173,True
3,PSMB7,0.000880,0.001173,True
4,RRM1,0.000880,0.001173,True
5,SF3B3,0.000880,0.001173,True
6,SKA2,0.007916,0.009046,True
7,TLN1,0.000880,0.001173,True


Seven of the eight genes are called, HAUS6 is not, and no pseudo-gene is called.

Every targeting guide cuts DNA and a non-targeting guide does not, so a call against non-targeting controls includes the response to the cut.
Aconcagua's library includes olfactory-receptor genes, which HeLa cells don't express, and passing them as `reference` separates the gene's effect from the cut.
The slice has none of them, and {doc}`hit calling methods <hit_calling_methods>` shows the difference on another screen.

The features with the largest effects say what changed.

In [17]:
top = boot.loc[boot["pvalue"] < 0.05].assign(size=lambda d: d["effect"].abs()).sort_values("size", ascending=False)
top.groupby("group").head(3)[["group", "feature", "effect"]].set_index("group").sort_index()

,feature,effect
group,,
DTL,Nuclei_Intensity_IntegratedIntensity_DAPI,7.874860
DTL,Cells_Intensity_IntegratedIntensity_DAPI,5.649233
DTL,Nuclei_Intensity_IntegratedIntensity_GH2AX,4.671523
HAUS6,Nuclei_AreaShape_FormFactor,-0.363714
HAUS6,Nuclei_AreaShape_Solidity,-0.320451
HAUS6,Nuclei_RadialDistribution_MeanFrac_DAPI_4of4,0.262087
ITGAV,Nuclei_Intensity_MeanIntensityEdge_TUBULIN,1.680722
ITGAV,Nuclei_Intensity_MeanIntensity_TUBULIN,1.673211
ITGAV,Cells_Intensity_StdIntensityEdge_PHALLOIDIN,1.423246


## Not covered

mantispy starts from measured cells, so it leaves out these parts of Brieflow:

- The image stages, from preprocessing and base calling to segmentation, feature extraction and merging.
- The classifier that labels cells as interphase or mitotic. mantispy reads the label from `obs["Metadata_Class"]`.
- Montages, cluster annotation with an LLM, and the sweep over Leiden resolutions. Call {func}`~mantispy.tl.cluster` once per resolution instead.
- The distance of each gene to the controls in PHATE's diffusion potential (`calculate_potential_to_nontargeting`). Every row's potential depends on every other row, so the distance isn't calibrated, and {func}`~mantispy.tl.hit_calling` tests genes instead.